In [ ]:
# Imports
import os
import sys
import subprocess
import matplotlib

from pathlib import Path

## aggregateBatteryOutput.py
Script for aggregating battery outputs in intervals.

In [ ]:
def aggregateBatteryOutput():
    tool = "./betterAggregateBattery.py"
    result = subprocess.run([
        sys.executable,
        str(tool),
        "-i", r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\electric_bus_2026-07-30-10-53-35_battery.xml",
        "-o", r"..\files\batteryAggregatedx1800.xml",
        "-t", "1800",
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print("STDERR:", result.stderr)
        print("STDOUT:", result.stdout)
        print("Return code:", result.returncode)
    else:
        print("aggregateBatteryOutput finished.")

aggregateBatteryOutput()

## plotXMLAttributes.py
Create multiple 2D-plots of 2 arbitrary attributes from on or more xml files aggregated by an third attribute (i.e. detector-id).

In [ ]:
def plotXMLAttributes():
    tool = (Path(os.environ["SUMO_HOME"]) / "tools/visualization/plotXMLAttributes.py").resolve()
    battery_file = r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\eBuS\files\batteryAggregatedx1800.xml"
    # Plot 1: energy consumed per bus over time
    result = subprocess.run([
        sys.executable, str(tool),
        "-x", "time",
        "-y", "energyConsumed",
        "--idattr", "id",
        "--xlabel", "time [s]",
        "--ylabel", "energy [Wh]",
        "--title", "Energy consumed per bus over time",
        "-o", r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\eBuS\visualisation\plots\batteryAggregated1800x1.png",
        battery_file,
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print("STDERR:", result.stderr)
        print("STDOUT:", result.stdout)
        print("Return code:", result.returncode)
    else:
        print("plotXMLAttributes finished.")

plotXMLAttributes()

In [ ]:
# Charging Events
def plotXMLAttributes():
    tool = (Path(os.environ["SUMO_HOME"]) / "tools/visualization/plotXMLAttributes.py").resolve()
    chargingstation_file = r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\electric_bus_2026-07-30-10-53-35_chargingstations.xml"
    # Plot 1: energy consumed per bus over time
    result = subprocess.run([
        sys.executable, str(tool),
        "-x", "chargingBegin",
        "-y", "totalEnergyChargedIntoVehicle",
        "--idattr", "chargingStationId",
        "--xlabel", "time [s]",
        "--ylabel", "energy [Wh]",
        "--title", "Charging Events over time",
        "-o", r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\eBuS\visualisation\plots\energyChargedScatter.png",
        chargingstation_file,
        "--scatterplot",
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print("STDERR:", result.stderr)
        print("STDOUT:", result.stdout)
        print("Return code:", result.returncode)
    else:
        print("plotStops finished.")

plotXMLAttributes()

## tripStatistics.py

This script is to calculate the global performance indices according to SUMO-based simulation results. The calculation functions are directly defined in this script.

In [ ]:
def tripStatistics():

    tool = (Path(os.environ["SUMO_HOME"]) / "tools/output/tripStatistics.py").resolve()
    result = subprocess.run([
        "python",
        f"{str(tool)}",
        "-t", r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\electric_bus_2026-07-29-10-34-54_tripinfo.xml",
        "-o", "../files/tripinfo.txt",
        "-e"
    ])
    print("tripStatistics finished.")

tripStatistics()

## computeStoppingPlaceUsage.py
This tool reads stop-output and tracks the number of stopped vehicles over time at stopping places (i.e. parkingArea). A distinct output file will be created for each stopping place.

In [ ]:
def computeStoppingPlaceUsage():

    tool = (Path(os.environ["SUMO_HOME"]) / "tools/output/computeStoppingPlaceUsage.py").resolve()
    result = subprocess.run([
        sys.executable,
        f"{str(tool)}",
        "-s", r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\electric_bus_2026-07-30-10-53-35_stopinfo.xml",
    ],
    capture_output=True,
    text=True,)

    if result.stdout:
        print("STDOUT:")
        print(result.stdout)
    if result.stderr:   
        print("STDERR:")
        print(result.stderr)
    else:
        print("computeStoppingPlaceUsage finished.")

computeStoppingPlaceUsage()

## plot_trajectories.py
Create plot of all trajectories obtained from a file generated through --fcd-output. This tool in particular is located in <SUMO_HOME>/tools.

In [ ]:
def plot_trajectories():

    tool = (Path(os.environ["SUMO_HOME"]) / "tools/plot_trajectories.py").resolve()
    result = subprocess.run([
        sys.executable,
        f"{str(tool)}",
        "-t", "xy",
        "-o", "../visualisation/plots/allLocations_output.png",
        r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\electric_bus_2026-07-29-16-07-33_fcdinfo.xml",
        "--scatterplot"
    ])
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        print("STDOUT:", result.stdout)
        print("Return code:", result.returncode)
    else:
        print("plot_trajectories finished.")

plot_trajectories()

## plotStops.py
Plot a public transport schedule (either planned or actual timings). 

In [ ]:
import os
import re
import subprocess
import sys
import xml.etree.ElementTree as ET
from pathlib import Path


def _get_first_veh_id(route_file):
    """Return the id of the first vehicle/trip/flow found in a SUMO route file."""
    for _, elem in ET.iterparse(route_file, events=("start",)):
        if elem.tag in ("route") and "id" in elem.attrib:
            return elem.attrib["id"]
    raise ValueError(f"No vehicle/trip/flow with an id found in {route_file}")


def plotStops():
    tool = (Path(os.environ["SUMO_HOME"]) / "tools/visualization/plotStops.py").resolve()

    route_file = r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\electric\e_routes.rou.xml"
    stops_file = r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\berlin_bus_stops.add.xml"

    veh_id = _get_first_veh_id(route_file)

    result = subprocess.run([
        sys.executable,
        str(tool),
        "-r", route_file,
        "-a", stops_file,
        "-i", veh_id,
        "-v",
        "--legend",
        "--filter-ids", "*",
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print("STDERR:", result.stderr)
        print("STDOUT:", result.stdout)
        print("Return code:", result.returncode)
    else:
        print("plotStops finished.")


plotStops()


## Powershell for Duarouter Route Fixing


In [ ]:
&'C:\Program Files (x86)\Eclipse\Sumo\bin\duarouter.exe' -n C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\berlin.net.xml -r C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\berlin.rou.gz --skip-new-routes --repair --ignore-errors --ptline-routing -o e_berlin.rou.xml